In [ ]:
!pip install -q transformers sentencepiece tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# =========================================================
# XLM-RoBERTa + Evidential Deep Learning + MC Dropout (COLAB)
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (COLAB)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Tamil"
RESULT_DIR = "/content/drive/MyDrive/results/TAMIL/RoBERT"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_tam_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_tam_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_tam_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(
    SarcasmDataset(train_df, tokenizer),
    batch_size=8,
    shuffle=True
)

dev_loader = DataLoader(
    SarcasmDataset(dev_df, tokenizer),
    batch_size=16
)

test_loader = DataLoader(
    SarcasmDataset(test_df, tokenizer),
    batch_size=16
)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY FROM ALPHA
# -------------------------
def accuracy_from_alpha(alpha, labels):
    probs = alpha / alpha.sum(dim=1, keepdim=True)
    preds = probs.argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialXLMR(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialXLMR(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )

        labels = batch["labels"].to(DEVICE)
        loss = evidential_loss(alpha, labels, NUM_CLASSES)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    # ---- VALIDATION ----
    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)
            loss = evidential_loss(alpha, labels, NUM_CLASSES)

            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    # ---- EARLY STOPPING ----
    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()   # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...


Epoch 5 | Train Loss: 0.2543, Train Acc: 0.8285 | Val Loss: 0.3057, Val Acc: 0.7932


Epoch 6 [VAL]: 100%|██████████| 423/423 [00:45<00:00,  9.33it/s]



Epoch 6 | Train Loss: 0.2317, Train Acc: 0.8475 | Val Loss: 0.3065, Val Acc: 0.7885


Epoch 7 [VAL]: 100%|██████████| 423/423 [00:45<00:00,  9.30it/s]



Epoch 7 | Train Loss: 0.2076, Train Acc: 0.8639 | Val Loss: 0.3284, Val Acc: 0.7900
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.7921647532252337
F1: 0.5417536534446764

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.91      0.87      6186
           1       0.66      0.46      0.54      2263

    accuracy                           0.79      8449
   macro avg       0.74      0.69      0.70      8449
weighted avg       0.78      0.79      0.78      8449

Average Predictive Uncertainty: 0.0032997075


DISTILL BERT


In [ ]:
# =========================================================
# DistilBERT-multilingual + Evidential DL + MC Dropout
# Google Drive + Colab READY
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (GOOGLE DRIVE)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Tamil"
RESULT_DIR = "/content/drive/MyDrive/results/TAMIL/DistillBERT"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_tam_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_tam_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_tam_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "distilbert-base-multilingual-cased"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=16, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=32)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=32)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY
# -------------------------
def accuracy_from_alpha(alpha, labels):
    preds = (alpha / alpha.sum(dim=1, keepdim=True)).argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialDistilBERT(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]  # [CLS]
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialDistilBERT(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )
        labels = batch["labels"].to(DEVICE)

        loss = evidential_loss(alpha, labels, NUM_CLASSES)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    # ---- VALIDATION ----
    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)
            loss = evidential_loss(alpha, labels, NUM_CLASSES)

            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    # ---- EARLY STOPPING ----
    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()  # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...


Epoch 3 | Train Loss: 0.2035, Train Acc: 0.8721 | Val Loss: 0.3194, Val Acc: 0.8029


Epoch 4 [VAL]: 100%|██████████| 212/212 [00:23<00:00,  9.00it/s]



Epoch 4 | Train Loss: 0.1471, Train Acc: 0.9134 | Val Loss: 0.3176, Val Acc: 0.7945


Epoch 5 [VAL]: 100%|██████████| 212/212 [00:23<00:00,  9.02it/s]



Epoch 5 | Train Loss: 0.1130, Train Acc: 0.9359 | Val Loss: 0.3474, Val Acc: 0.7955
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.7986743993371996
F1: 0.6013592688071244

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.88      0.87      6186
           1       0.64      0.57      0.60      2263

    accuracy                           0.80      8449
   macro avg       0.74      0.73      0.73      8449
weighted avg       0.79      0.80      0.79      8449

Average Predictive Uncertainty: 0.0018955356


mBART

In [ ]:
# =========================================================
# mBART (facebook/mbart-large-50) + Evidential DL + MC Dropout
# Google Drive + Colab READY
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (GOOGLE DRIVE)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Tamil"
RESULT_DIR = "/content/drive/MyDrive/results/TAMIL/mBART"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_tam_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_tam_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_tam_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "facebook/mbart-large-50"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=4, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=8)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=8)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY
# -------------------------
def accuracy_from_alpha(alpha, labels):
    preds = (alpha / alpha.sum(dim=1, keepdim=True)).argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialMBART(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.d_model, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]   # <s>
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialMBART(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )
        labels = batch["labels"].to(DEVICE)

        loss = evidential_loss(alpha, labels, NUM_CLASSES)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    # ---- VALIDATION ----
    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)

            loss = evidential_loss(alpha, labels, NUM_CLASSES)
            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    # ---- EARLY STOPPING ----
    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()   # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


Epoch 1 [VAL]: 100%|██████████| 845/845 [00:51<00:00, 16.52it/s]



Epoch 1 | Train Loss: 0.3615, Train Acc: 0.7431 | Val Loss: 0.3076, Val Acc: 0.7737


Epoch 2 [VAL]: 100%|██████████| 845/845 [00:51<00:00, 16.56it/s]



Epoch 2 | Train Loss: 0.2759, Train Acc: 0.8017 | Val Loss: 0.2835, Val Acc: 0.7957


...


Epoch 4 | Train Loss: 0.1967, Train Acc: 0.8691 | Val Loss: 0.2960, Val Acc: 0.7904


Epoch 5 [VAL]: 100%|██████████| 845/845 [00:51<00:00, 16.56it/s]



Epoch 5 | Train Loss: 0.1668, Train Acc: 0.8931 | Val Loss: 0.3359, Val Acc: 0.7848


Epoch 6 [VAL]: 100%|██████████| 845/845 [00:51<00:00, 16.53it/s]



Epoch 6 | Train Loss: 0.1372, Train Acc: 0.9158 | Val Loss: 0.3522, Val Acc: 0.7861
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.7937033968516984
F1: 0.5244201909959072

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.93      0.87      6186
           1       0.69      0.42      0.52      2263

    accuracy                           0.79      8449
   macro avg       0.75      0.68      0.70      8449
weighted avg       0.78      0.79      0.78      8449

Average Predictive Uncertainty: 0.0015331205


DeBERTa

In [ ]:
# =========================================================
# mDeBERTa-v3 + Evidential Deep Learning + MC Dropout
# Google Drive + Colab READY
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (GOOGLE DRIVE)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Tamil"
RESULT_DIR = "/content/drive/MyDrive/results/TAMIL/DeBERT"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_tam_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_tam_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_tam_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "microsoft/mdeberta-v3-base"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=8, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=16)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=16)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY
# -------------------------
def accuracy_from_alpha(alpha, labels):
    preds = (alpha / alpha.sum(dim=1, keepdim=True)).argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialDeBERTa(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]   # CLS
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialDeBERTa(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )
        labels = batch["labels"].to(DEVICE)

        loss = evidential_loss(alpha, labels, NUM_CLASSES)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    # ---- VALIDATION ----
    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)

            loss = evidential_loss(alpha, labels, NUM_CLASSES)
            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    # ---- EARLY STOPPING ----
    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()   # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Epoch 1 [VAL]: 100%|██████████| 423/423 [00:17<00:00, 23.74it/s]



Epoch 1 | Train Loss: 0.3529, Train Acc: 0.7481 | Val Loss: 0.3109, Val Acc: 0.7805


Epoch 2 [VAL]: 100%|██████████| 423/423 [00:17<00:00, 23.67it/s]



Epoch 2 | Train Loss: 0.2864, Train Acc: 0.8027 | Val Loss: 0.2919, Val Acc: 0.8007


...


Epoch 5 | Train Loss: 0.1582, Train Acc: 0.9045 | Val Loss: 0.3273, Val Acc: 0.7970


Epoch 6 [VAL]: 100%|██████████| 423/423 [00:17<00:00, 23.84it/s]



Epoch 6 | Train Loss: 0.1320, Train Acc: 0.9219 | Val Loss: 0.3322, Val Acc: 0.7955


Epoch 7 [VAL]: 100%|██████████| 423/423 [00:17<00:00, 23.75it/s]



Epoch 7 | Train Loss: 0.1103, Train Acc: 0.9365 | Val Loss: 0.3356, Val Acc: 0.8012
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.7990294709433069
F1: 0.5828009828009828

Classification Report:

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      6186
           1       0.66      0.52      0.58      2263

    accuracy                           0.80      8449
   macro avg       0.75      0.71      0.73      8449
weighted avg       0.79      0.80      0.79      8449

Average Predictive Uncertainty: 0.006318171


BERT


In [ ]:
# =========================================================
# Multilingual BERT + Evidential DL + MC Dropout
# Google Drive + Colab READY
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (GOOGLE DRIVE)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Tamil"
RESULT_DIR = "/content/drive/MyDrive/results/TAMIL/BERT"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_tam_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_tam_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_tam_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "bert-base-multilingual-cased"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=16, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=32)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=32)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY
# -------------------------
def accuracy_from_alpha(alpha, labels):
    preds = (alpha / alpha.sum(dim=1, keepdim=True)).argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialBERT(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]   # [CLS]
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialBERT(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )
        labels = batch["labels"].to(DEVICE)

        loss = evidential_loss(alpha, labels, NUM_CLASSES)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)

            loss = evidential_loss(alpha, labels, NUM_CLASSES)
            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()   # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...


Epoch 2 | Train Loss: 0.3390, Train Acc: 0.7576 | Val Loss: 0.3432, Val Acc: 0.7491


Epoch 3 [VAL]: 100%|██████████| 212/212 [00:12<00:00, 16.46it/s]



Epoch 3 | Train Loss: 0.3351, Train Acc: 0.7610 | Val Loss: 0.3326, Val Acc: 0.7471


Epoch 4 [VAL]: 100%|██████████| 212/212 [00:12<00:00, 16.44it/s]



Epoch 4 | Train Loss: 0.3353, Train Acc: 0.7704 | Val Loss: 0.3479, Val Acc: 0.7505
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.7431648715824358
F1: 0.4247083775185578

Classification Report:

              precision    recall  f1-score   support

           0       0.79      0.89      0.83      6186
           1       0.53      0.35      0.42      2263

    accuracy                           0.74      8449
   macro avg       0.66      0.62      0.63      8449
weighted avg       0.72      0.74      0.72      8449

Average Predictive Uncertainty: 0.00090523844


MiniLM

In [ ]:
# =========================================================
# MiniLM Multilingual + Evidential DL + MC Dropout
# Google Drive + Colab READY
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (GOOGLE DRIVE)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Tamil"
RESULT_DIR = "/content/drive/MyDrive/results/TAMIL/MiniLM"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_tam_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_tam_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_tam_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "microsoft/multilingual-MiniLM-L12-H384"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=32, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=64)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=64)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY
# -------------------------
def accuracy_from_alpha(alpha, labels):
    preds = (alpha / alpha.sum(dim=1, keepdim=True)).argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialMiniLM(nn.Module):
    def __init__(self, num_classes=2, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]   # [CLS]
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialMiniLM(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )
        labels = batch["labels"].to(DEVICE)

        loss = evidential_loss(alpha, labels, NUM_CLASSES)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)

            loss = evidential_loss(alpha, labels, NUM_CLASSES)
            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()   # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...


Epoch 6 | Train Loss: 0.3175, Train Acc: 0.7908 | Val Loss: 0.3347, Val Acc: 0.7775


Epoch 7 [VAL]: 100%|██████████| 106/106 [00:05<00:00, 19.66it/s]



Epoch 7 | Train Loss: 0.3035, Train Acc: 0.8036 | Val Loss: 0.3242, Val Acc: 0.7811


Epoch 8 [VAL]: 100%|██████████| 106/106 [00:05<00:00, 19.76it/s]



Epoch 8 | Train Loss: 0.3542, Train Acc: 0.7567 | Val Loss: 0.3804, Val Acc: 0.7310
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.7716889572730501
F1: 0.4742436631234669

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.91      0.85      6186
           1       0.62      0.38      0.47      2263

    accuracy                           0.77      8449
   macro avg       0.71      0.65      0.66      8449
weighted avg       0.75      0.77      0.75      8449

Average Predictive Uncertainty: 0.0023109266
